# PatchCore Training - Master Batch Pipeline
## Optimized for MVTec AD Categories

- ✅ Automated loop for all remaining categories
- ✅ `bottle` excluded as it is already trained
- ✅ Memory optimized for Kaggle (`gc.collect()` and `empty_cache()`)

In [ ]:
# Install required packages
!pip install torch torchvision timm scikit-learn pillow numpy tqdm

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import roc_auc_score
import time
from datetime import datetime
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration
We have excluded `bottle` because you have already successfully trained it.

In [ ]:
# CONFIGURATION - Professional ML Pipeline

# Excluded 'bottle' as it is already trained
CATEGORIES = [
    'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 
    'leather', 'metal_nut', 'pill', 'screw', 'tile', 
    'toothbrush', 'transistor', 'wood', 'zipper'
]

# Base Kaggle paths
DATASET_BASE_DIR = '/kaggle/input/mvtec-ad'
OUTPUT_DIR = '/kaggle/working/models'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training parameters
CORESET_RATIO = 0.1      # Keep 10% of patches
NUM_NEIGHBORS = 1        # K for KNN
SIGMA = 2.0              # Threshold multiplier
INPUT_SIZE = (224, 224)  # Image size

## Load Backbone Model & Preprocessing Methods
Loaded only ONCE for all categories to save memory and time.

In [ ]:
# Load WideResNet50 backbone
print("Loading WideResNet50 backbone...")
backbone = models.wide_resnet50_2(pretrained=True)
backbone.eval()
backbone.to(device)

features = {}
def make_hook(name):
    def hook(module, input, output):
        features[name] = output.detach()
    return hook

backbone.layer2.register_forward_hook(make_hook('layer2'))
backbone.layer3.register_forward_hook(make_hook('layer3'))

avg_pool = torch.nn.AvgPool2d(3, stride=1, padding=1)
print("✅ Backbone loaded")

# Preprocessing
transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_image(img_path):
    img = Image.open(img_path).convert('RGB')
    return transform(img).unsqueeze(0)

def extract_patches(img_path):
    img_tensor = load_image(img_path).to(device)
    with torch.no_grad():
        backbone(img_tensor)
    
    l2 = avg_pool(features['layer2'])
    l3 = avg_pool(features['layer3'])
    l3 = F.interpolate(l3, size=l2.shape[2:], mode='bilinear', align_corners=False)
    
    embedding = torch.cat([l2, l3], dim=1)
    b, c, h, w = embedding.shape
    patches = embedding.reshape(c, h * w).T
    return patches.cpu().numpy()

def greedy_coreset(features, ratio):
    n = features.shape[0]
    target = max(1, int(n * ratio))
    if target >= n: return features
    
    rng = np.random.default_rng(42)
    selected = [int(rng.integers(n))]
    min_dists = np.full(n, np.inf)
    
    for _ in tqdm(range(target - 1), desc="  Coreset", leave=False):
        last = features[selected[-1]]
        dists = np.linalg.norm(features - last, axis=1)
        min_dists = np.minimum(min_dists, dists)
        selected.append(int(np.argmax(min_dists)))
    return features[selected]

def score_image(img_path, memory_bank, num_neighbors):
    patches = extract_patches(img_path)
    patches_tensor = torch.tensor(patches, dtype=torch.float32, device=device)
    dists = torch.cdist(patches_tensor, memory_bank, p=2.0)
    topk, _ = dists.topk(num_neighbors, dim=1, largest=False)
    avg_dists = topk.mean(dim=1)
    return float(avg_dists.max().item())


## Master Batch Training Loop

In [ ]:
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting Batch Training for {len(CATEGORIES)} categories...")
results_summary = []

for idx, category in enumerate(CATEGORIES):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(CATEGORIES)}] 🚀 TRAINING CATEGORY: {category.upper()}")
    print(f"{'='*60}")
    start_time = time.time()
    
    train_dir = Path(DATASET_BASE_DIR) / category / 'train' / 'good'
    test_dir = Path(DATASET_BASE_DIR) / category / 'test'
    output_file = Path(OUTPUT_DIR) / f"{category}_patchcore_fixed.pkl"
    
    train_images = list(train_dir.glob('*.png')) + list(train_dir.glob('*.jpg'))
    if len(train_images) == 0:
        print(f"⚠️ WARNING: No images found for {category} in {train_dir}. Skipping...")
        continue
    
    print(f"  Found {len(train_images)} training images.")
    
    # 1. Extract Features
    print("  [1/4] Extracting patch embeddings...")
    all_patches = []
    for img_path in tqdm(train_images, desc=f"  {category} Embedding", leave=False):
        try:
            all_patches.append(extract_patches(img_path))
        except Exception as e:
            print(f"  Skipped {img_path.name}: {e}")
            
    all_patches_np = np.concatenate(all_patches, axis=0)
    print(f"    Total patches: {all_patches_np.shape[0]:,}")

    # 2. Coreset Subsampling
    print("  [2/4] Coreset subsampling...")
    memory_bank_np = greedy_coreset(all_patches_np, CORESET_RATIO)
    memory_bank = torch.tensor(memory_bank_np, dtype=torch.float32, device=device)
    print(f"    Memory bank size: {memory_bank_np.shape[0]:,} patches")

    # 3. Score Normal Images for Threshold
    print("  [3/4] Computing p99_normal and threshold...")
    raw_distances = []
    for img_path in tqdm(train_images, desc=f"  {category} Scoring", leave=False):
        try:
            raw_distances.append(score_image(img_path, memory_bank, NUM_NEIGHBORS))
        except Exception as e:
            print(f"  Skipped {img_path.name}: {e}")

    raw_distances = np.array(raw_distances)
    p99_normal = float(np.percentile(raw_distances, 99))
    normalized_scores = raw_distances / p99_normal
    mean_score = float(np.mean(normalized_scores))
    std_score = float(np.std(normalized_scores))
    threshold = round(max(1.0, mean_score + SIGMA * std_score), 6)

    print(f"    p99_normal: {p99_normal:.4f}")
    print(f"    Threshold:  {threshold:.4f}")

    # 4. Evaluate Test Set
    auroc = None
    if test_dir.exists():
        print("  [4/4] Evaluating on test set...")
        labels, scores = [], []
        for subdir in sorted(test_dir.iterdir()):
            if not subdir.is_dir(): continue
            label = 0 if subdir.name == 'good' else 1
            images = list(subdir.glob('*.png')) + list(subdir.glob('*.jpg'))
            for img_path in tqdm(images, desc=f"    Test {subdir.name}", leave=False):
                try:
                    scores.append(score_image(img_path, memory_bank, NUM_NEIGHBORS) / p99_normal)
                    labels.append(label)
                except:
                    pass
        if labels:
            auroc = roc_auc_score(labels, scores)
            print(f"    ✅ AUROC: {auroc:.4f}")

    # 5. Save Model
    print(f"  Saving model to {output_file.name}...")
    model_data = {
        'memory_bank': memory_bank_np,
        'threshold': threshold,
        'num_neighbors': NUM_NEIGHBORS,
        'p99_normal': p99_normal,
        'raw_distances': raw_distances.tolist(),
        'normalized_scores': normalized_scores.tolist(),
        'config': {
            'category': category,
            'backbone': 'wide_resnet50_2',
            'layers': ['layer2', 'layer3'],
            'coreset_ratio': CORESET_RATIO,
            'sigma': SIGMA,
            'i_auroc': auroc,
            'p99_normal': p99_normal
        }
    }
    with open(output_file, 'wb') as f:
        pickle.dump(model_data, f)
        
    duration = (time.time() - start_time) / 60
    print(f"  ✅ Completed {category} in {duration:.1f} minutes.")
    
    results_summary.append({
        'category': category,
        'auroc': round(auroc, 4) if auroc else None,
        'time_mins': round(duration, 1),
        'threshold': threshold
    })
    
    del all_patches, all_patches_np, memory_bank, memory_bank_np, model_data
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("🏆 BATCH TRAINING COMPLETE")
print(f"{'='*60}")
for res in results_summary:
    print(f"{res['category'].ljust(15)} | AUROC: {res['auroc']} | Time: {res['time_mins']}m | Threshold: {res['threshold']}")
